# Draft search walkthrough

---

These cells use the 2025 NFL draft and save Chroma to disk.

1. Pick a dataset
2. Chunk and embed your data
3. Store your data in a Chroma vector database
4. Build a search app with Gradio
5. Publish on Hugging Face Spaces (done outside this notebook; see the README)

The source file is the full nflverse draft table:

https://github.com/nflverse/nflverse-data/releases/download/draft_picks/draft_picks.parquet

That file starts in 1980. The build script keeps one season with `load_draft_picks(2025)`. This notebook does not download anything. It reads the saved copy of those 257 rows, `data/raw/draft-picks-2025.csv`.

Start the notebook from the project folder or from `notebooks/`. Install once, from the project folder (the notebook also needs a Jupyter kernel, such as `pip install ipykernel`):

`pip install -r requirements.txt`

## 1. Load the embedding model

The model is loaded first because the chunk, embed, and search cells all use it. The first run downloads it. Later runs reuse the local copy. The build script and `app.py` must keep this exact name.

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


## 2. Pick a dataset

Read the raw table. Each row is one drafted player. Position is still a short code such as `QB`.

In [2]:
import polars as pl
from pathlib import Path

# Find the project root. This works from the project folder, from notebooks/,
# or from any direct child such as scripts/. A deeper folder will not find the CSV.
root = Path.cwd()
if not (root / "data" / "raw" / "draft-picks-2025.csv").exists():
    root = root.parent

raw_path = root / "data" / "raw" / "draft-picks-2025.csv"
draft = pl.read_csv(raw_path)

print(draft.height)
print(
    draft.select(
        ["season", "round", "pick", "team", "pfr_player_name", "position", "college"]
    ).head(3)
)


257
shape: (3, 7)
┌────────┬───────┬──────┬──────┬─────────────────┬──────────┬────────────┐
│ season ┆ round ┆ pick ┆ team ┆ pfr_player_name ┆ position ┆ college    │
│ ---    ┆ ---   ┆ ---  ┆ ---  ┆ ---             ┆ ---      ┆ ---        │
│ i64    ┆ i64   ┆ i64  ┆ str  ┆ str             ┆ str      ┆ str        │
╞════════╪═══════╪══════╪══════╪═════════════════╪══════════╪════════════╡
│ 2025   ┆ 1     ┆ 1    ┆ TEN  ┆ Cam Ward        ┆ QB       ┆ Miami (FL) │
│ 2025   ┆ 1     ┆ 2    ┆ JAX  ┆ Travis Hunter   ┆ WR       ┆ Colorado   │
│ 2025   ┆ 1     ┆ 3    ┆ NYG  ┆ Abdul Carter    ┆ DE       ┆ Penn St.   │
└────────┴───────┴──────┴──────┴─────────────────┴──────────┴────────────┘


## 3. Chunk

A chunk is the piece of text that Chroma stores and returns. Each draft pick is short, so one pick is one chunk and no text is split. The chunk is one sentence.

The sentence uses the full position word, because the model reads ordinary words better than codes such as `QB`. The position goes first and is repeated, because the wording after it is the same for every player.

In [3]:
POSITION_WORDS = {
    "QB": "quarterback",
    "RB": "running back",
    "WR": "wide receiver",
    "TE": "tight end",
    "OL": "offensive lineman",
    "OT": "offensive tackle",
    "OG": "offensive guard",
    "G": "offensive guard",
    "C": "center",
    "DL": "defensive lineman",
    "DE": "defensive end",
    "DT": "defensive tackle",
    "LB": "linebacker",
    "OLB": "outside linebacker",
    "DB": "defensive back",
    "CB": "cornerback",
    "S": "safety",
    "SAF": "safety",
    "FS": "free safety",
    "K": "kicker",
    "P": "punter",
    "LS": "long snapper",
}

# Round numbers as words, so a query like "first round" can match.
ROUND_WORDS = {
    1: "first",
    2: "second",
    3: "third",
    4: "fourth",
    5: "fifth",
    6: "sixth",
    7: "seventh",
}

texts = []
ids = []
for row in draft.iter_rows(named=True):
    name = row["pfr_player_name"] or "Unknown player"
    code = row["position"] or ""
    college = row["college"] or "unknown college"
    team = row["team"] or "an unknown team"
    position = POSITION_WORDS.get(code, code or "unknown position")
    round_word = ROUND_WORDS.get(row["round"], "unknown")

    sentence = (
        f"{position}. {name} is a {position} from {college}. "
        f"{team} drafted him in the {round_word} round of the "
        f"{row['season']} NFL draft, pick {row['pick']}."
    )
    texts.append(sentence)
    # String id. 2025-1 is the first overall pick.
    ids.append(f"{row['season']}-{row['pick']}")

processed_path = root / "data" / "processed" / "indexed-draft-sentences-2025.csv"
pl.DataFrame({"id": ids, "document": texts}).write_csv(processed_path)

print(len(texts))
print(texts[0])


257
quarterback. Cam Ward is a quarterback from Miami (FL). TEN drafted him in the first round of the 2025 NFL draft, pick 1.


## 4. Embed

`encode` turns each sentence into a vector of 384 numbers. `normalize_embeddings=True` scales each list of numbers to length 1. The search cell does the same to the question, so questions and sentences are treated alike. A smaller distance means a closer match.

In [4]:
embeddings = model.encode(texts, normalize_embeddings=True)
print(embeddings.shape)


(257, 384)


## 5. Store in Chroma

An in-memory database disappears when the notebook stops. `PersistentClient` writes `chromadb/chroma.sqlite3` so `app.py` can open it later.

Running this cell again replaces the `draft_2025` collection.

In [5]:
import chromadb
from chromadb.config import Settings
from pathlib import Path

# Same project-root check as the dataset cell.
root = Path.cwd()
if not (root / "data" / "raw" / "draft-picks-2025.csv").exists():
    root = root.parent

client = chromadb.PersistentClient(
    path=str(root / "chromadb"),
    settings=Settings(anonymized_telemetry=False),
)

# Delete first. Adding the same ids again would skip the new sentences.
try:
    client.delete_collection("draft_2025")
except Exception:
    pass

collection = client.create_collection(
    "draft_2025",
    # Smaller cosine distance means a closer pick.
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=ids,
)

print(collection.count())


257


## 6. Search

Embed the question with the same model, then ask Chroma for the nearest sentences. This search returns the five closest results. The number is set by `n_results`.

In [6]:
def semantic_search(query):
    query = (query or "").strip()
    if not query:
        return "Type a search, such as: quarterback."

    # One vector for the question, normalized like the stored picks.
    query_embedding = model.encode([query], normalize_embeddings=True)
    # n_results is fixed at 5 here. app.py reads the same number from
    # draft_search_settings.py (N_RESULTS).
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5,
    )

    lines = []
    for document, distance in zip(results["documents"][0], results["distances"][0]):
        lines.append(f"{distance:.3f} — {document}")
    return "\n\n".join(lines)


print(semantic_search("quarterback"))


0.365 — quarterback. Cam Ward is a quarterback from Miami (FL). TEN drafted him in the first round of the 2025 NFL draft, pick 1.

0.388 — quarterback. Tyler Shough is a quarterback from Louisville. NOR drafted him in the second round of the 2025 NFL draft, pick 40.

0.415 — quarterback. Jalen Milroe is a quarterback from Alabama. SEA drafted him in the third round of the 2025 NFL draft, pick 92.

0.422 — quarterback. Graham Mertz is a quarterback from Florida. HOU drafted him in the sixth round of the 2025 NFL draft, pick 197.

0.434 — quarterback. Jaxson Dart is a quarterback from Mississippi. NYG drafted him in the first round of the 2025 NFL draft, pick 25.


## 7. Gradio

One text box in, one text box out. This cell opens a local page. If port 7860 is already in use, stop the other `app.py` first.

In [ ]:
# Build the same page as app.py: one input box, one output box, four example
# questions. launch() keeps this cell running until you stop it. Stop it
# before running the Chroma cell again.
import gradio as gr

demo = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(label="Enter your search query"),
    outputs=gr.Textbox(label="Top Matches"),
    title="Semantic Search Engine",
    description=(
        "Search the 2025 NFL draft. "
        "Source: https://github.com/nflverse/nflverse-data/releases/download/draft_picks/draft_picks.parquet. "
        "A smaller distance is a closer match."
    ),
    examples=["quarterback", "wide receiver", "kicker", "defensive end"],
)

demo.launch()


## 8. Where those steps landed

The notebook cells above are the same work as `scripts/build-draft-index.py` and `app.py`. Those two files read their shared settings from `draft_search_settings.py`. This notebook keeps its own copies of the model name, the collection name, and the result count, so they must match that file.

- `data/raw/draft-picks-2025.csv` is the dataset cell
- `data/processed/indexed-draft-sentences-2025.csv` is the chunk cell
- `chromadb/chroma.sqlite3` is the store cell
- `app.py` is the search function plus the Gradio cell, loading the saved database instead of building it
- `requirements.txt` is the install line
- `README.md` holds the Hugging Face Space settings at the top and the project overview below them

Run the app outside the notebook with `python app.py`.

Publishing (assignment step 5) is not a notebook cell. See the README and `docs/search-app-and-space.md` for the upload steps.